In [ ]:
from nn2 import *
i = 64
Rms = lambda*_:Res(RMSNorm(i), *_)
CPA = lambda:(
    Rms(Add(
        Cnv(i, g=4., r=(3,3)),
        Pol(i, h=1.),
        At(i, h=1., G=4, g=1, r=(12,11)),
        Swi(i, h=4.),
    )),
    Rms(Add(
        Cnv(i, g=4., r=(3,3)),
        Pol(i, h=1.),
        At(i, h=1., G=4, g=1, r=(12,11)),
        Swi(i, h=4.),
    )),
    Rms(Add(
        Cnv(i, g=4., r=(3,3)),
        Pol(i, h=1.),
        At(i, h=1., G=4, g=1, r=(12,11)),
        Swi(i, h=4.),
    )),
)
mdl = Seq(
    Zip(
        Embedding(256, i),
        Linear(i, i, 0),
        Linear(i, i, 0)
    ),
    Seq(
        Rms(Swi(i, h=4.)),
        *(_:=lambda n:[
            *CPA(),
            Rms(
                Linear(i, i*2, 0),
                T(MaxPool1d(2)),
                Linear(i*2, i, 0),
                *_(n-1),
                At(i, h=1., G=4, g=1, o=2., r=(12,11)),
                Unflatten(2, (2,-1)),
                Flatten(1, 2)
            ),
            *CPA()
        ]if n else CPA())(7)
    ),
    Swi(i, 4., 256),
).to('cuda',torch.bfloat16)
mdl[1] = torch.compile(mdl[1], fullgraph=True, dynamic=False)
opt = torch.optim.AdamW(mdl.parameters(), fused=True)
lr = torch.optim.lr_scheduler.OneCycleLR(opt, 5e-3, 100000, cycle_momentum=False)
plot = Plot()
for x,t in Dl():
    if lr.last_epoch == lr.total_steps: break
    opt.zero_grad()
    x = mdl[:2](x)
    y = mdl[2](x),
    y += mdl([y[0].argmax(2), x, y[0].softmax(2)@mdl[0][0].weight]),
    l = [functional.cross_entropy(y.mT,t)for y in y]
    reduce(torch.add,l).backward()
    opt.step()
    lr.step()
    plot(l=torch.stack(l).data, a=torch.stack([(y.argmax(2)==t).sum()/t.numel()for y in y]))